In [11]:
using JuMP
using GLMakie
using LinearAlgebra
using DifferentialEquations
using Plots
using Ipopt
using ForwardDiff

In [5]:
@kwdef struct CartpoleParameters
cart_mass::Float64 = 1.0
pole_mass::Float64 = 0.2
pole_length::Float64 = 0.5
gravity::Float64 = 9.81
damping::Float64 = 0.05
maximum_force::Float64 = 10.0
track_limit::Float64 = 2.4
end

const POSITION = 1
const ANGLE = 2
const VELOCITY = 3
const ANGULAR_VELOCITY = 4
const FORCE = 1


1

In [ ]:
function cartpole_dynamics(x, u, p, t)
    q, θ, q̇, θ̇ = x
    force = u[FORCE]
    s = sin(θ)
    c = cos(θ)
    denominator = p.cart_mass + p.pole_mass * s^2
    q̈ = (
        force - p.damping * q̇ +
        p.pole_mass * s * (p.pole_length * θ̇^2 + p.gravity * c)
    ) / denominator

    θ̈ = -(q̈ * c + p.gravity * s) / p.pole_length
    return [q̇, θ̇, q̈, θ̈]
end

function zero_dev(x, u, p, t)
    dyn = cartpole_dynamics(x,u,p,t)

    if (dyn[1] == 0 && dyn[2] == 0)
        return "H"
    else
        return "N"
    end
end


zero_dev (generic function with 1 method)

In [ ]:
N = 81
duration = 4.0
times = LinRange(0, duration, N)
dt = times[2] - times[1]
p = CartpoleParameters()

#These are state vectors that have index [x, θ, ẋ, θ̇]
initial_state = [0; 0; 0; 0]
goal_state = [0; π; 0; 0]

(nx, nu) = (4, 1)
model = Model(Ipopt.Optimizer)
set_optimizer_attribute(model, "max_iter", 300) # Don't run too long

@variable(model, X[1:nx, 1:N])
@variable(model, -p.maximum_force .<=U[1:nu, 1:N-1] .<= p.maximum_force)
@constraint(model, -p.track_limit .<=X[1,:].<=p.track_limit)

for i in 1:nx
    fix(X[i, 1], initial_state[i]; force = true)
    fix(X[i, N], goal_state[i]; force = true)
end

for k in 1:(N - 1)
    predicted_state = X[:,k] + dt * cartpole_dynamics(X[:, k],U[:, k],p,times[k])
    for i in 1:nx
        @constraint(model, X[i, k + 1] == predicted_state[i])
    end
end
@objective(model, Min, sum(U .^2))
optimize!(model)

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.9.0.

Number of nonzeros in equality constraint Jacobian...:     1266
Number of nonzeros in inequality constraint Jacobian.:       79
Number of nonzeros in Lagrangian Hessian.............:     1188

Total number of variables............................:      396
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       80
                     variables with only upper bounds:        0
Total number of equality constraints.................:      320
Total number of inequality constraints...............:       81
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:       81
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 3.14e+00 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
function draw_cartpole(state, p::CartpoleParameters)
    q, θ = state[POSITION], state[ANGLE]
    tip = Point2f(
        q + p.pole_length * sin(θ),
        -p.pole_length * cos(θ),
    )

    ax = Axis(
        fig[1, 1];
        xlabel = "cart position",
        ylabel = "height",
        aspect = DataAspect(),
        limits = (
            -p.track_limit - 0.3,
            p.track_limit + 0.3,
            -p.pole_length - 0.2,
            p.pole_length + 0.3,
        ),
    )
    lines!(ax, [-p.track_limit, p.track_limit], [0, 0]; linewidth = 2)
    lines!(ax, [Point2f(q, 0), tip]; linewidth = 4)
    CairoMakie.scatter!(ax, [Point2f(q, 0), tip]; markersize = [20, 14])

    return fig
end



function draw_cartpole_2(fig, state, p::CartpoleParameters)

    q, θ = state[POSITION], state[ANGLE]
    tip = Point2f(
        q + p.pole_length * sin(θ),
        -p.pole_length * cos(θ),
    )

    empty!(fig)
    ax = Axis(
        fig[1, 1];
        xlabel = "cart position",
        ylabel = "height",
        aspect = DataAspect(),
        limits = (
            -p.track_limit - 0.3,
            p.track_limit + 0.3,
            -p.pole_length - 0.2,
            p.pole_length + 0.3,
        ),
    )

    lines!(ax, [-p.track_limit, p.track_limit], [0, 0]; linewidth = 2)
    lines!(ax, [Point2f(q, 0), tip]; linewidth = 4)
    CairoMakie.scatter!(ax, [Point2f(q, 0), tip]; markersize = [20, 14])

    return fig
end


vals = value.(X)
fig = draw_cartpole(initial_state, p)

record(fig, "Cart_Pole.mp4", 1:N, framerate = 20) do idx
    draw_cartpole_2(fig, vals[:, idx], p)
end


"Cart_Pole_3.mp4"